In [1]:
%matplotlib inline
%config InlineBackend.figure_formats = ['svg']

import json
import yaml

import os
import subprocess
import sys
import time

import tempfile
import h5py

import pandas as pd
import numpy as np
import matplotlib as mpl

import matplotlib.pyplot as plt

import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import h5py

plt.rcParams['axes.prop_cycle'] = plt.cycler(color=['olivedrab', 'steelblue', 'firebrick', 'goldenrod'])
plt.rcParams['axes.formatter.use_mathtext'] = True
plt.rcParams['axes.formatter.useoffset'] = False
plt.rcParams['axes.formatter.limits'] = (0, 0)
plt.rcParams['figure.figsize'] = [6,4]
plt.rcParams['figure.constrained_layout.use'] = True
plt.rcParams['legend.frameon'] = False
plt.rcParams['xtick.minor.visible'] = True
plt.rcParams['ytick.minor.visible'] = True

template_file = 'PSLS/examples/psls.yaml'

with open(template_file, 'r') as file:
    config_dict = yaml.safe_load(file)
    print(json.dumps(config_dict, indent=4, sort_keys=False))

{
    "Observation": {
        "QuarterDuration": [
            90.0,
            90.0,
            90.0
        ],
        "MasterSeed": 1704040900,
        "Gaps": {
            "Enable": 1,
            "Seed": -1,
            "InterQuarterGapDuration": 3.0,
            "RandomGapDuration": 0.0,
            "RandomGapTimeFraction": 0.5,
            "RandomGapStep": 0.0,
            "PeriodicGapCadence": 5.0,
            "PeriodicGapDuration": 20.0,
            "PeriodicGapJitter": 2.0,
            "PeriodicGapStep": 0.0
        }
    },
    "Instrument": {
        "Sampling": 25.0,
        "IntegrationTime": 21.0,
        "GroupID": [
            1,
            2,
            3,
            4
        ],
        "NCamera": 6,
        "TimeShift": 6.25,
        "RandomNoise": {
            "Enable": 1,
            "Type": "PLATO_SIMU",
            "NSR": 73.0
        },
        "Systematics": {
            "Enable": 1,
            "Table": "systematics/PLATO_systematics_BOL_V2.npy",
  

In [2]:
psls_template = {
    'Observation': {
        'QuarterDuration': [90.0, 90.0, 90.0],
        'MasterSeed': 1704040900,
        'Gaps': {
            'Enable': 1,
            'Seed': -1,
            'InterQuarterGapDuration': 3.0,
            'RandomGapDuration': 0.0,
            'RandomGapTimeFraction': 0.5,
            'RandomGapStep': 0.0,
            'PeriodicGapCadence': 5.0,
            'PeriodicGapDuration': 20.0,
            'PeriodicGapJitter': 2.0,
            'PeriodicGapStep': 0.0
        }
    },
    'Instrument': {
        'Sampling': 25.0,
        'IntegrationTime': 21.0,
        'GroupID': [
            1,
            2,
            3,
            4
        ],
        'NCamera': 6,
        'TimeShift': 6.25,
        'RandomNoise': {
            'Enable': 1,
            'Type': 'PLATO_SIMU',
            'NSR': 73.0
        },
        'Systematics': {
            'Enable': 1,
            'Table': 'systematics/PLATO_systematics_BOL_V2.npy',
            'Version': 2,
            'DriftLevel': 'any',
            'Seed': -1
        }
    },
    'Star': {
        'Mag': 10.0,
        'ID': 12069449,
        'ModelType': 'single',
        'ModelDir': 'models/',
        'ModelName': '0012069449',
        'ES': 'ms',
        'Teff': 5750.0,
        'Logg': 4.353,
        'SurfaceRotationPeriod': 0.0,
        'CoreRotationFreq': 0.0,
        'Inclination': 0.0
    },
    'Oscillations': {
        'Enable': 1,
        'numax': 179.3,
        'delta_nu': 13.68,
        'DPI': 80.58,
        'q': 0.15,
        'SurfaceEffects': 1,
        'Seed': -1
    },
    'Activity': {
        'Enable': 1,
        'Sigma': 40.0,
        'Tau': 0.2,
        'Seed': -1,
        'Spot': {
            'Enable': 0,
            'dOmega': 0.0,
            'MuStar': 0.59,
            'MuSpot': 0.78,
            'Radius': [2.5, 2.5, 2.5],
            'Latitude': [0.0, 20.0, 40.0],
            'Longitude': [0.0, 0.0, 0.0],
            'Lifetime': [10, 30, 50],
            'TimeMax': [-1, -1, -1],
            'Contrast': [0.7, 0.8, 0.6],
            'Modulation': 0.0,
            'Seed': -1
        },
        'Flare': {
            'Enable': 0,
            'MeanPeriod': 2,
            'Amplitude': 2500.0,
            'UpDown': 0.1,
            'MeanDuration': -1,
            'DurationDispersion': -1,
            'Seed': -1
        }
    },
    'Granulation': {
        'Enable': 1,
        'Type': 1,
        'Seed': -1
    },
    'Transit': {
        'Enable': 1,
        'PlanetRadius': 0.5,
        'OrbitalPeriod': 10.0,
        'PlanetSemiMajorAxis': 1.0,
        'OrbitalAngle': 0.0,
        'LimbDarkeningCoefficients': [0.25, 0.75]
    },
    'External': {
        'Enable': 0,
        'FilePath': 'examples/external_example.txt'
    }
}

In [3]:
%%time

def generate_lightcurve():
    with tempfile.NamedTemporaryFile(suffix='.yaml', dir='PSLS', mode='w', delete=True) as tf:
        yaml.dump(psls_template, tf, default_flow_style=False, sort_keys=False)
        tf.flush()
        
        print(f'--> run simulation with {tf.name}:')
        print()
        print('%cd ', end='')
        %cd PSLS
        print()
        !echo '' | ./psls.py -o data -V {tf.name}
        print('\n')
        print('%cd ', end='')
        %cd ..
        print()

generate_lightcurve()

--> run simulation with /home/fritz/work/notebooks/MASS/Astrobiology/project/PSLS/tmpfeaayz0x.yaml:

%cd /home/fritz/work/notebooks/MASS/Astrobiology/project/PSLS

Star name: 0012069449
Surface effects parameters, a = -0.004719 ,b = 5.136143
V magnitude: 10.000000
P magnitude: 9.617556
Reference V PLATO magnitude (6000 K): 9.957556
Total white-noise [ppm.Hz^(-1/2)]: 0.000000e+00
Total white-noise at sampling time: 0.000000e+00
mode properties load from models/0012069449.gsm
numax = 2556.537235, deltanu = 116.850299 [muHz]
------------------------------------------------------------------------
('Amax=', 4.057202727671265, ' [ppm]')
('Gamma=', 1.1126892540304767, ' [muHz]')
('Hmax=', 9.418020832292957, ' [ppm^2/muHz]')
('Width=', 9.418020832292957, ' [muHz]')
theoretical input frequencies:
n, l, nu_eig, nu_var , nu_eig-nu_var,nu_eig-nu_richardson inertia
1 0 222.916477 222.881039 0.035438 -0.010603 4.525142e-04
2 0 357.224806 357.182072 0.042734 0.000343 7.940757e-05
3 0 473.326188 473.

In [13]:
config_files = [psls_template, psls_template, psls_template, psls_template, psls_template]

total_sims = len(config_files)

initial_status = f'progress |{'░' * 50}| 0% (running 1/{total_sims})          \n'

sys.stdout.write(f'\r{initial_status}')
sys.stdout.flush()

for i, cfg in enumerate(config_files):
    if i != 0 and i != total_sims - 1:
        perc = i / total_sims * 100
        bar = '█' * int(perc // 2) + '░' * (50 - int(perc // 2))
        status = f'progress |{bar}| {perc:3.0f}% (running {i+1}/{total_sims})          \n'

        sys.stdout.write(f'\r{status}')
        sys.stdout.flush()

    with tempfile.NamedTemporaryFile(suffix='.yaml', dir='PSLS', mode='w', delete=True) as tf:
        yaml.dump(cfg, tf, default_flow_style=False, sort_keys=False)
        tf.flush()

        print('    sampling parameters [done]')
        print('    generating lightcurve [done]')
        
        %cd -q PSLS
        !echo '' | ./psls.py -o data {tf.name}
        %cd -q ..

        print('    saving dataframe [done]')

#        process = subprocess.Popen(
#            ['./psls.py', '-o', 'data', '-V', tf.name],
#            stdin=subprocess.PIPE,
#            stdout=subprocess.PIPE,
#            stderr=subprocess.STDOUT,
#            text=True,
#            cwd='PSLS'
#        )

#        for line in process.stdout:
#            clean_log = line.strip()
#            # We truncate the log to 60 chars to keep the line tidy
#            status_text = f'[{perc:5.1f}%] Run {i+1}/{total_sims} | Activity: {clean_log[:60]}'
#            
#            # \r moves the cursor to the start of the line; end='' prevents a new line
#            sys.stdout.write(f'\r{status_text: <100}')
#            sys.stdout.flush()

#        process.communicate(input='\n')

final_status = f'progress |{'█' * 50}| 100% (finished {i+1}/{total_sims})          '

sys.stdout.write(f'\r{final_status}')
sys.stdout.flush()

progress |░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░| 0% (running 1/5)          
    sampling parameters [done]
    generating lightcurve [done]
    saving dataframe [done]
progress |██████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░|  20% (running 2/5)          
    sampling parameters [done]
    generating lightcurve [done]
    saving dataframe [done]
progress |████████████████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░|  40% (running 3/5)          
    sampling parameters [done]
    generating lightcurve [done]
    saving dataframe [done]
progress |██████████████████████████████░░░░░░░░░░░░░░░░░░░░|  60% (running 4/5)          
    sampling parameters [done]
    generating lightcurve [done]
    saving dataframe [done]
    sampling parameters [done]
    generating lightcurve [done]
    saving dataframe [done]
progress |██████████████████████████████████████████████████| 100% (finished 5/5)          